# Configure Cloud Kafka Connection (Confluent Cloud) 100 file

In [ ]:
# Step 1: Install required libraries in the cloud environment
!pip install confluent_kafka pandas

import os
import json
import time
import pandas as pd
from confluent_kafka import Producer
from google.colab import drive

# Step 2: Mount your Google Drive to access the shortcut folder
drive.mount('/content/drive')

# Step 3: Configure Cloud Kafka Connection (Confluent Cloud)
conf = {
    'bootstrap.servers': 'pkc-921jm.us-east-2.aws.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': 'SKIUYYELSQ4EMY4T',
    'sasl.password': 'cfltLdHPgkUVVBKUuf5nF4aCTH8vF+Ft1yhVjjvk0J9TTmnByjFmoAD0PDWSbcCg'
}

# Change the topic name to create a completely new and clean pipeline
producer = Producer(conf)
topic_name = 'smart-grid-city-data'

# Callback function to log delivery status of messages
def delivery_report(err, msg):
    if err is not None:
        pass # To avoid cluttering the screen with messages when streaming 100 houses
    # Printing for each message is disabled to speed up performance; we will only print when a file is fully processed

# Step 4: Set the path to the shared shortcut folder in your Drive
folder_path = '/content/drive/MyDrive/Project DEPI/realistic_multi_house_raw/'

if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' was not found.")
else:
    all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    print(f"Found {len(all_files)} smart meter files in the directory. Starting massive ingestion...")

    # Step 5: Loop through all files and stream rows
    for file_name in all_files:
        file_path = os.path.join(folder_path, file_name)
        print(f"Streaming data from: {file_name} ...", end=" ")

        df = pd.read_csv(file_path)

        for index, row in df.iterrows():
            data_payload = row.to_dict()
            data_payload['meter_id'] = file_name.replace('.csv', '')

            json_data = json.dumps(data_payload)

            producer.produce(topic_name, value=json_data.encode('utf-8'), callback=delivery_report)
            producer.poll(0)

            # Significantly speed up the streaming to handle the volume of 100 meters
            time.sleep(0.01)

        print("[DONE]") # Print only when the entire file is fully processed

    # Step 6: Flush any remaining messages
    producer.flush()
    print("Finished streaming ALL 100 meter files to the cloud cluster successfully!")